# CIFAR-10 Starter: Download + Cache 10,000 Images

This notebook downloads CIFAR-10 once, builds a local cache with exactly 10,000 images, and reuses that cache on future runs.

In [2]:
from pathlib import Path
import tarfile
import pickle
import urllib.request

import numpy as np

CIFAR10_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
CACHE_DIR = DATA_DIR / "cache"

ARCHIVE_PATH = RAW_DIR / "cifar-10-python.tar.gz"
EXTRACTED_DIR = RAW_DIR / "cifar-10-batches-py"
CACHE_PATH = CACHE_DIR / "cifar10_10000.npz"

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

RAW_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def _safe_extract(tar: tarfile.TarFile, path: Path) -> None:
    base = path.resolve()
    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(base)):
            raise RuntimeError(f"Unsafe path in tar file: {member.name}")
    tar.extractall(path=path)


def download_and_extract_cifar10() -> None:
    if EXTRACTED_DIR.exists():
        print(f"Dataset already extracted at: {EXTRACTED_DIR}")
        return

    if not ARCHIVE_PATH.exists():
        print("Downloading CIFAR-10 archive...")
        urllib.request.urlretrieve(CIFAR10_URL, ARCHIVE_PATH)
        print(f"Saved archive to: {ARCHIVE_PATH}")
    else:
        print(f"Archive already exists at: {ARCHIVE_PATH}")

    print("Extracting archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        _safe_extract(tar, RAW_DIR)
    print(f"Extracted to: {EXTRACTED_DIR}")


def load_cifar_batch(batch_file: Path):
    with batch_file.open("rb") as f:
        batch = pickle.load(f, encoding="bytes")

    images = batch[b"data"]
    labels = np.array(batch[b"labels"], dtype=np.int64)

    images = images.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return images, labels

In [4]:
if CACHE_PATH.exists():
    print(f"Loading cached dataset from: {CACHE_PATH}")
    with np.load(CACHE_PATH) as cached:
        images = cached["images"]
        labels = cached["labels"]
        class_names = cached["class_names"].tolist()
else:
    print("Cache not found. Building cache from CIFAR-10...")
    download_and_extract_cifar10()

    # data_batch_1 contains exactly 10,000 training images
    batch_file = EXTRACTED_DIR / "data_batch_1"
    images, labels = load_cifar_batch(batch_file)
    class_names = CLASS_NAMES

    np.savez_compressed(
        CACHE_PATH,
        images=images.astype(np.uint8),
        labels=labels.astype(np.int64),
        class_names=np.array(class_names),
    )
    print(f"Cache saved to: {CACHE_PATH}")

print("Done.")
print(f"images shape: {images.shape}")
print(f"labels shape: {labels.shape}")
print(f"unique classes: {len(set(labels.tolist()))}")

Cache not found. Building cache from CIFAR-10...
Saved archive to: data\raw\cifar-10-python.tar.gz
Extracting archive...


C:\Users\Istamosh\AppData\Local\Temp\ipykernel_912\3478800765.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=path)


Extracted to: data\raw\cifar-10-batches-py


C:\Users\Istamosh\AppData\Local\Temp\ipykernel_912\3478800765.py:30: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  batch = pickle.load(f, encoding="bytes")


Cache saved to: data\cache\cifar10_10000.npz
Done.
images shape: (10000, 32, 32, 3)
labels shape: (10000,)
unique classes: 10


In [5]:
# Quick sanity check
print("First 10 labels:", labels[:10].tolist())
print("First label name:", class_names[int(labels[0])])

First 10 labels: [6, 9, 9, 4, 1, 1, 2, 7, 8, 3]
First label name: frog
